In [11]:
import pandas as pd
import requests
import time
from requests.adapters import HTTPAdapter, Retry

In [12]:
# Load your metadata Excel file
df = pd.read_excel("metadata.xlsx")
commands = []

In [13]:
# Use a persistent session with retries for ENA API calls
session = requests.Session()
retries = Retry(total=5, backoff_factor=0.3)
session.mount("https://", HTTPAdapter(max_retries=retries))

def get_fastq_links(run_accession):
    """Query ENA API for fastq_ftp links for a given run accession."""
    url = (
        "https://www.ebi.ac.uk/ena/portal/api/filereport"
        f"?accession={run_accession}&result=read_run&fields=fastq_ftp&format=tsv"
    )
    try:
        r = session.get(url, timeout=10)
        if r.status_code != 200:
            print(f"⚠️ ENA API error for {run_accession}")
            return []
        lines = r.text.strip().split("\n")
        if len(lines) > 1 and lines[1].strip():
            # split by TAB, then split by ;
            fields = lines[1].split("\t")
            fastq_field = fields[-1]
            return fastq_field.split(";")
        return []
    except Exception as e:
        print(f"⚠️ Error fetching links for {run_accession}: {e}")
        return []

In [ ]:
# RUN 1: 38m 4.0s at 7-22-2025 2:18 pm 
# RUN 2: 37m 25s
for idx, row in df.iterrows():
    country = str(row["country"]).replace(" ", "_")
    isolate = str(row["isolate name"]).replace(" ", "_")
    ena_run = str(row["ena_run"]).strip()
    ena_experiment = str(row["ena_experiment"]).strip()
    ena_sample = str(row["ena_sample"]).strip()
    aux_link = str(row.get("auxillary.ftp.link")).strip()

    # Case 1: If auxillary link is provided, use it directly
    if pd.notnull(aux_link) and aux_link and aux_link.lower() != "nan":
        # Ensure ftp:// prefix
        if not aux_link.startswith("ftp://"):
            link = f"ftp://{aux_link}"
        else:
            link = aux_link
        output_name = f"{country}_{isolate}.fastq.gz"
        cmd = f"aria2c -x 8 -s 8 \"{link}\" -o \"{output_name}\""
        commands.append(cmd)

    # Case 2: If a valid ENA accession is provided, fetch real fastq links
    elif (
        (pd.notnull(ena_run) and ena_run.startswith(("ERR", "SRR", "SAM", "ERX"))) or
        (pd.notnull(ena_experiment) and ena_experiment.startswith(("ERX", "ERS"))) or
        (pd.notnull(ena_sample) and ena_sample.startswith(("ERS", "SAM")))
    ):
        # Pick the first valid accession
        accession = None
        if pd.notnull(ena_run) and ena_run.startswith(("ERR", "SRR", "SAM", "ERX")):
            accession = ena_run
        elif pd.notnull(ena_experiment) and ena_experiment.startswith(("ERX", "ERS")):
            accession = ena_experiment
        elif pd.notnull(ena_sample) and ena_sample.startswith(("ERS", "SAM")):
            accession = ena_sample

        if accession:
            links = get_fastq_links(accession)
            time.sleep(0.2) 
            if links:
                for link in links:
                    # Ensure ftp:// prefix
                    if not link.startswith("ftp://"):
                        link = f"ftp://{link}"
                    link_file = link.split("/")[-1]
                    output_name = f"{country}_{accession}_{link_file}"
                    cmd = f"aria2c -x 8 -s 8 \"{link}\" -o \"{output_name}\""
                    commands.append(cmd)
            else:
                print(f"⚠️ No fastq_ftp found for {accession}")
        else:
            print(f"⚠️ No valid accession found for row {idx}: {row.to_dict()}")

In [ ]:
with open("download_fastqs.sh", "w") as f:
    for cmd in commands:
        f.write(cmd + "\n")

print(f"✅ Done: download_fastqs.sh ready with {len(commands)} commands.")
# then use cat download_fastqs.sh | parallel --bar -j 8

# started at 3:13 pm

✅ Done: download_fastqs.sh ready with 10109 commands.
